## 0. Install packages

In [ ]:
!pip install -qU langchain langchain-community langchain-text-splitters \
    langchain-experimental langchain-groq langchain-neo4j neo4j pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

## 1. PDF Extraction

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "AI_in_Healthcare_Report.pdf"   # <-- change to your uploaded file's name

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print("Number of pages:", len(docs))
print("\nPreview of page 1:\n")
print(docs[0].page_content[:1000])


/tmp/ipykernel_1566/3917312355.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of pages: 20

Preview of page 1:

Artificial Intelligence
in Healthcare
Market Landscape, Clinical Applications,
and Strategic Outlook
PREPARED FOR
Strategy & Innovation Review
September 2026    Illustrative Report with Sample Data


In [ ]:
#print all data
print(docs)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-04T05:56:33+00:00', 'author': 'Strategy & Innovation Review', 'keywords': '', 'moddate': '2026-09-04T05:56:33+00:00', 'subject': '(unspecified)', 'title': 'Artificial Intelligence in Healthcare', 'trapped': '/False', 'source': 'AI_in_Healthcare_Report.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1'}, page_content='Artificial Intelligence\nin Healthcare\nMarket Landscape, Clinical Applications,\nand Strategic Outlook\nPREPARED FOR\nStrategy & Innovation Review\nSeptember 2026  \x7f  Illustrative Report with Sample Data'), Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-04T05:56:33+00:00', 'author': 'Strategy & Innovation Review', 'keywords': '', 'moddate': '2026-09-04T05:56:33+00:00', 'subject': '(unspecified)', 'title': 'Artificial Intelligence in Healthcare', 'trapped': '/False', '

## 2. Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(docs)
print("Number of chunks:", len(chunks))


Number of chunks: 34


## 3. LLM Graph Transformation

In [ ]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
print("Groq key loaded:", bool(GROQ_API_KEY))


Groq key loaded: True


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY,
)

# quick connection check
print(llm.invoke("Say: connection successful").content)


connection successful


In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(llm=llm, strict_mode=True)
print("Graph transformer ready.")


Graph transformer ready.


## 4. Neo4j Database

In [ ]:
NEO4J_URI = userdata.get("NEO4J_URI")
NEO4J_USERNAME = userdata.get("NEO4J_USERNAME")
NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")
NEO4J_DATABASE = userdata.get("NEO4J_DATABASE")

from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)

print(graph.query("RETURN 'Connected!' AS message"))


[{'message': 'Connected!'}]


In [ ]:
graph.add_graph_documents(all_graph_documents, include_source=True)

print(graph.query("MATCH (n) RETURN count(n) AS total_nodes"))
print(graph.query("MATCH ()-[r]->() RETURN count(r) AS total_relationships"))


[{'total_nodes': 526}]
[{'total_relationships': 980}]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english")
chunk_matrix = vectorizer.fit_transform([c.page_content for c in chunks])

def search_chunks(query, k=4):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, chunk_matrix)[0]
    top_idx = scores.argsort()[::-1][:k]
    return [(chunks[i], scores[i]) for i in top_idx]

# quick check
for chunk, score in search_chunks("What are the barriers to AI adoption?", k=2):
    print(f"[{score:.3f}] {chunk.page_content[:100]}...")


[0.316] 09  Barriers and Challenges
Despite strong momentum, healthcare organizations continue to report sig...
[0.192] Table of Contents
1
Executive Summary
 3
2
Introduction
 4
3
Market Landscape and Growth Projections...


## 5. Retrieval

Two steps, kept simple:
1. Ask the LLM which entities the question is about.
2. Look up those entities' relationships in Neo4j and hand them to the LLM as context
   for the final answer.


In [ ]:
def extract_entities(question):
    prompt = f"""List only the key entities or concepts in this question,
as a comma-separated list, nothing else.

Question: {question}
Entities:"""
    raw = llm.invoke(prompt).content.strip()
    return [e.strip() for e in raw.split(",") if e.strip()]


def retrieve_from_graph(entities, limit_per_entity=15):
    triples = []
    for entity in entities:
        result = graph.query(
            """
            MATCH (n)-[r]-(m)
            WHERE (toLower(n.id) CONTAINS toLower($entity) OR toLower(m.id) CONTAINS toLower($entity))
              AND NOT type(r) = 'MENTIONS'
            RETURN DISTINCT n.id AS source, type(r) AS relationship, m.id AS target
            LIMIT $limit
            """,
            params={"entity": entity, "limit": limit_per_entity},
        )
        triples.extend(result)
    return triples


def answer_question(question, k_chunks=4):
    # --- graph side ---
    entities = extract_entities(question)
    triples = retrieve_from_graph(entities)
    graph_context = "\n".join(
        f"{t['source']} --[{t['relationship']}]--> {t['target']}" for t in triples
    ) or "(no relevant relationships found in the graph)"

    # --- vector side ---
    top_chunks = search_chunks(question, k=k_chunks)
    chunk_context = "\n\n".join(c.page_content for c, _ in top_chunks) or "(no relevant chunks found)"

    prompt = f"""Answer the question using BOTH sources below. Prefer the knowledge
graph for precise factual relationships, and the text passages for supporting detail
and nuance. If neither source has enough information, say so clearly.

Question: {question}

Text passages:
{chunk_context}

Knowledge graph:
{graph_context}

Give a clear, concise answer.
"""
    answer = llm.invoke(prompt).content
    return {
        "question": question,
        "entities": entities,
        "triples": triples,
        "chunks": top_chunks,
        "answer": answer,
    }


questions = [
    "What are the applications of AI in healthcare?",
    "What are the barriers to AI adoption?",
    "Which regions have the highest AI adoption?",
    "What is the time to value for medical imaging?",
]

for q in questions:
    result = answer_question(q)
    print("QUESTION:", result["question"])
    print("Entities:", result["entities"])
    print("Graph triples found:", len(result["triples"]))
    print("Chunks retrieved:", len(result["chunks"]))
    print("ANSWER:", result["answer"])
    print("-" * 80)


QUESTION: What are the applications of AI in healthcare?
Entities: ['AI', 'healthcare', 'applications']
Graph triples found: 45
Chunks retrieved: 4
ANSWER: **Applications of AI in healthcare**

| AI application segment | What it does (from the text) | Key facts (knowledge‑graph relationships) |
|------------------------|------------------------------|------------------------------------------|
| **Medical imaging** | Computer‑vision models help radiologists spot abnormalities in X‑ray, CT, and MRI scans, often as a triage or second‑reader tool. | `Medical Imaging` `IS_SEGMENT_OF` `Ai Applications`. |
| **Diagnostics** | Broad diagnostic‑support tools, including pathology slide analysis and AI‑assisted differential diagnosis. | `Diagnostics` `IS_SEGMENT_OF` `Ai Applications`. |
| **Drug discovery** | Machine‑learning models speed up target identification, molecule screening, and clinical‑trial design. | `Drug Discovery` `IS_SEGMENT_OF` `Ai Applications`. |
| **Virtual health assistants*

## 6. Evaluation

A small set of question/reference-answer pairs, scored by the LLM itself
(correctness 1–5, faithfulness 1–5). Replace the questions and reference answers
with ones drawn from your own PDF.


In [ ]:
EVAL_SET = [
    {
        "question": "What are the main barriers to scaling AI adoption in healthcare?",
        "reference_answer": "Data privacy and security, integration with legacy IT systems, governance, clinician trust, and workforce skill gaps are major barriers to scaling AI adoption.",
    },
    {
        "question": "Which region has the highest AI adoption share in healthcare?",
        "reference_answer": "North America has the highest AI adoption share at 42%.",
    },
    {
        "question": "What are the most mature AI applications in healthcare?",
        "reference_answer": "Medical imaging and diagnostics are among the most mature and best-evidenced AI applications in healthcare.",
    },
    {
        "question": "What was the projected global AI-in-healthcare market size for 2026?",
        "reference_answer": "The projected global AI-in-healthcare market size for 2026 was $36.1 billion.",
    },
    {
        "question": "What improvement in diagnostic accuracy has AI shown over average clinician benchmarks?",
        "reference_answer": "AI-assisted diagnostic tools showed accuracy improvements of approximately 4–9 percentage points over average clinician benchmarks in select studies.",
    },
    {
        "question": "What are the main benefits of AI adoption in healthcare?",
        "reference_answer": "Benefits include improved diagnostic accuracy, clinician time savings, increased patient throughput, earlier detection, and cost reduction.",
    },

]
print(f"{len(EVAL_SET)} evaluation questions loaded.")

6 evaluation questions loaded.


In [ ]:
import json, re

def judge_answer(question, reference_answer, candidate_answer):
    prompt = f"""You are grading an AI-generated answer against a reference answer.

Question: {question}
Reference answer: {reference_answer}
Candidate answer: {candidate_answer}

Score the candidate from 1 (poor) to 5 (excellent) on:
- "correctness": does it convey the same facts as the reference answer?
- "faithfulness": is it a reasonable, grounded answer (no made-up facts)?

Respond ONLY with JSON: {{"correctness": <int>, "faithfulness": <int>}}
"""
    raw = llm.invoke(prompt).content.strip().replace("```json", "").replace("```", "")
    try:
        return json.loads(raw)
    except Exception:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        return json.loads(match.group(0)) if match else {"correctness": None, "faithfulness": None}


In [ ]:
import pandas as pd

rows = []
for item in EVAL_SET:
    result = answer_question(item["question"])
    scores = judge_answer(item["question"], item["reference_answer"], result["answer"])
    rows.append({
        "question": item["question"],
        "answer": result["answer"],
        "triples_used": len(result["triples"]),
        "correctness": scores.get("correctness"),
        "faithfulness": scores.get("faithfulness"),
    })

eval_df = pd.DataFrame(rows)
eval_df


,question,answer,triples_used,correctness,faithfulness
0,What are the main barriers to scaling AI adopt...,**Main barriers to scaling AI in healthcare**\...,48,5,5
1,Which region has the highest AI adoption share...,The region with the highest AI adoption share ...,30,5,5
2,What are the most mature AI applications in he...,**Most mature AI applications in healthcare**\...,47,5,5
3,What was the projected global AI-in-healthcare...,The projected global AI‑in‑healthcare market s...,15,1,1
4,What improvement in diagnostic accuracy has AI...,AI‑assisted diagnostic tools have been shown t...,27,5,5
5,What are the main benefits of AI adoption in h...,**Main benefits of AI adoption in healthcare**...,38,5,4


In [ ]:
print("Average faithfulness:", eval_df["faithfulness"].mean())


Average correctness: 4.333333333333333
Average faithfulness: 4.166666666666667


## Load Data from Checkpoint

In [ ]:
import pickle

pkl_file_path = 'graph_documents_checkpoint.pkl'

with open(pkl_file_path, 'rb') as f:
    loaded_graph_documents = pickle.load(f)

print(f"Successfully loaded data from '{pkl_file_path}'.")
print(f"Type of loaded data: {type(loaded_graph_documents)}")

# Display a snippet of the loaded data for verification
if isinstance(loaded_graph_documents, list) and len(loaded_graph_documents) > 0:
    print("First element of loaded data:")
    display(loaded_graph_documents[0])
else:
    display(loaded_graph_documents)

Successfully loaded data from 'graph_documents_checkpoint.pkl'.
Type of loaded data: <class 'dict'>


{'graph_documents': [GraphDocument(nodes=[Node(id='Artificial Intelligence In Healthcare', type='Concept', properties={}), Node(id='Market Landscape', type='Concept', properties={}), Node(id='Clinical Applications', type='Concept', properties={}), Node(id='Strategic Outlook', type='Concept', properties={}), Node(id='Strategy & Innovation Review', type='Organization', properties={}), Node(id='September 2026', type='Date', properties={}), Node(id='Illustrative Report With Sample Data', type='Document', properties={})], relationships=[Relationship(source=Node(id='Illustrative Report With Sample Data', type='Document', properties={}), target=Node(id='Market Landscape', type='Concept', properties={}), type='TOPIC', properties={}), Relationship(source=Node(id='Illustrative Report With Sample Data', type='Document', properties={}), target=Node(id='Clinical Applications', type='Concept', properties={}), type='TOPIC', properties={}), Relationship(source=Node(id='Illustrative Report With Sample 